In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import glob, os, random
from PIL import Image
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve, auc
)
from sklearn.preprocessing import label_binarize
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split

device     = 'cuda' if torch.cuda.is_available() else 'cpu'
CLASS_NAMES = ['1st Degree', '2nd Degree', '3rd Degree']
CHECKPOINT  = 'best_burn_model.pth'
AUGMENTED_DATA_ROOT = os.path.join('..', 'data_augmented')
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

print('Device:', device)


Device: cuda


## 1 — Load model

In [3]:
def build_model(num_classes=3, dropout_p=0.4):
    weights = models.EfficientNet_B3_Weights.DEFAULT
    model   = models.efficientnet_b3(weights=weights)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=dropout_p),
        nn.Linear(in_features, 512),
        nn.SiLU(),
        nn.Dropout(p=dropout_p * 0.5),
        nn.Linear(512, num_classes)
    )
    return model


model = build_model()
model.load_state_dict(torch.load(CHECKPOINT, map_location=device))
model = model.to(device)
model.eval()
print(f'Model loaded from {CHECKPOINT}')


FileNotFoundError: [Errno 2] No such file or directory: 'best_burn_model.pth'

## 2 — Rebuild test set  
Same split as training (same seed) so test images are unseen.

In [ ]:
CLASS_MAP = {'1st degree': 0, '2nd degree': 1, '3rd degree': 2}
all_paths, all_labels = [], []
for cls_name, lbl in CLASS_MAP.items():
    paths = glob.glob(os.path.join(AUGMENTED_DATA_ROOT, cls_name, '*'))
    all_paths.extend(paths)
    all_labels.extend([lbl] * len(paths))

data = pd.DataFrame({'image_path': all_paths, 'label': all_labels})
data = data.sample(frac=1, random_state=42).reset_index(drop=True)

_, temp = train_test_split(data, test_size=0.2, stratify=data['label'], random_state=42)
_, test_data = train_test_split(temp, test_size=0.5, stratify=temp['label'], random_state=42)
test_data = test_data.reset_index(drop=True)

print(f'Test set: {len(test_data)} images')
print(test_data['label'].value_counts().sort_index())


## 3 — Transforms & dataset

In [ ]:
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

tta_transforms = [
    val_transforms,
    transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]),
    transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]),
    transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomRotation(degrees=(90, 90)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]),
]


class BurnDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform
    def __len__(self):
        return len(self.dataframe)
    def __getitem__(self, idx):
        row   = self.dataframe.iloc[idx]
        image = Image.open(row['image_path']).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, int(row['label'])


test_dataset = BurnDataset(test_data, transform=val_transforms)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0, pin_memory=True)


## 4 — Standard evaluation (no TTA)

In [ ]:
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images.to(device))
        probs   = F.softmax(outputs, dim=1).cpu().numpy()
        preds   = outputs.argmax(1).cpu().numpy()
        all_probs.extend(probs)
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

all_probs  = np.array(all_probs)
all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

print('=' * 55)
print('TEST RESULTS — no TTA')
print('=' * 55)
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))


## 5 — TTA evaluation

In [ ]:
def tta_predict(model, dataset, tta_transforms_list):
    model.eval()
    all_probs, all_labels = [], []
    for idx in range(len(dataset)):
        row   = dataset.dataframe.iloc[idx]
        image = Image.open(row['image_path']).convert('RGB')
        label = int(row['label'])
        aug_probs = []
        for tfm in tta_transforms_list:
            tensor = tfm(image).unsqueeze(0).to(device)
            with torch.no_grad():
                prob = F.softmax(model(tensor), dim=1).squeeze().cpu().numpy()
            aug_probs.append(prob)
        all_probs.append(np.mean(aug_probs, axis=0))
        all_labels.append(label)
    return np.array(all_probs), np.array(all_labels)


tta_probs, tta_labels = tta_predict(model, test_dataset, tta_transforms)
tta_preds = np.argmax(tta_probs, axis=1)

print('=' * 55)
print('TEST RESULTS — with TTA')
print('=' * 55)
print(classification_report(tta_labels, tta_preds, target_names=CLASS_NAMES))


## 6 — Confusion matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, preds, title in [
    (axes[0], all_preds,  'No TTA'),
    (axes[1], tta_preds,  'With TTA'),
]:
    cm = confusion_matrix(all_labels, preds)
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES).plot(
        ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title)

plt.tight_layout()
plt.savefig('confusion_matrix_test.png', dpi=150)
plt.show()


## 7 — ROC curves (one-vs-rest per class)

In [ ]:
y_bin = label_binarize(tta_labels, classes=[0, 1, 2])
colors = ['#378ADD', '#EF9F27', '#1D9E75']

fig, ax = plt.subplots(figsize=(7, 6))
for i, (cls, color) in enumerate(zip(CLASS_NAMES, colors)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], tta_probs[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{cls}  AUC={roc_auc:.3f}')

ax.plot([0,1],[0,1],'--',color='#888780',lw=1)
ax.set_xlabel('False positive rate')
ax.set_ylabel('True positive rate')
ax.set_title('ROC curves — one-vs-rest (TTA)')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150)
plt.show()


## 8 — Confidence distribution per class

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ['#378ADD', '#EF9F27', '#1D9E75']

for i, (cls, color) in enumerate(zip(CLASS_NAMES, colors)):
    mask        = tta_labels == i
    correct_conf = tta_probs[mask & (tta_preds == i), i]
    wrong_conf   = tta_probs[mask & (tta_preds != i), i]
    axes[i].hist(correct_conf, bins=20, alpha=0.7, color=color,   label=f'Correct ({len(correct_conf)})')
    axes[i].hist(wrong_conf,   bins=20, alpha=0.7, color='#E24B4A', label=f'Wrong   ({len(wrong_conf)})')
    axes[i].set_title(cls)
    axes[i].set_xlabel('Confidence')
    axes[i].set_ylabel('Count')
    axes[i].legend(fontsize=9)
    axes[i].grid(True, alpha=0.3)

plt.suptitle('Confidence distribution — correct vs wrong predictions', fontsize=12)
plt.tight_layout()
plt.savefig('confidence_distribution.png', dpi=150)
plt.show()


## 9 — Misclassified samples

In [ ]:
wrong_idx = np.where(tta_preds != tta_labels)[0]
print(f'Total misclassified: {len(wrong_idx)} / {len(tta_labels)}  ({len(wrong_idx)/len(tta_labels)*100:.1f}%)')

sample_idx = random.sample(list(wrong_idx), min(12, len(wrong_idx)))
fig, axes  = plt.subplots(2, 6, figsize=(18, 6))
axes = axes.flatten()

for plot_i, idx in enumerate(sample_idx):
    row   = test_data.iloc[idx]
    img   = Image.open(row['image_path']).convert('RGB')
    true  = CLASS_NAMES[tta_labels[idx]]
    pred  = CLASS_NAMES[tta_preds[idx]]
    conf  = tta_probs[idx, tta_preds[idx]]
    axes[plot_i].imshow(img)
    axes[plot_i].set_title(f'True: {true}\nPred: {pred} ({conf:.0%})', fontsize=8, color='#A32D2D')
    axes[plot_i].axis('off')

plt.suptitle('Misclassified samples (random 12)', fontsize=12)
plt.tight_layout()
plt.savefig('misclassified_samples.png', dpi=120)
plt.show()


## 10 — Per-class threshold tuning (fixes 2nd degree recall)

In [ ]:
def apply_thresholds(probs, thresholds):
    adjusted = probs * np.array(thresholds)
    return np.argmax(adjusted, axis=1)


best_thresholds = [1.0, 1.0, 1.0]
best_f1_2nd     = 0.0
best_overall    = 0.0

from sklearn.metrics import f1_score

for t2 in np.arange(1.0, 2.0, 0.05):
    thresholds = [1.0, round(t2, 2), 1.0]
    preds      = apply_thresholds(tta_probs, thresholds)
    f1_2nd     = f1_score(tta_labels, preds, average=None)[1]
    f1_overall = f1_score(tta_labels, preds, average='macro')
    if f1_2nd > best_f1_2nd and f1_overall >= best_overall - 0.01:
        best_f1_2nd     = f1_2nd
        best_overall    = f1_overall
        best_thresholds = thresholds

print(f'Best thresholds found: {best_thresholds}')
tuned_preds = apply_thresholds(tta_probs, best_thresholds)

print('\n' + '=' * 55)
print('RESULTS — TTA + threshold tuning')
print('=' * 55)
print(classification_report(tta_labels, tuned_preds, target_names=CLASS_NAMES))


## 11 — Single image prediction function

In [ ]:
def predict_single(img_path, use_tta=True, thresholds=None):
    image = Image.open(img_path).convert('RGB')

    if use_tta:
        probs_list = []
        for tfm in tta_transforms:
            tensor = tfm(image).unsqueeze(0).to(device)
            with torch.no_grad():
                probs_list.append(F.softmax(model(tensor), dim=1).squeeze().cpu().numpy())
        probs = np.mean(probs_list, axis=0)
    else:
        tensor = val_transforms(image).unsqueeze(0).to(device)
        with torch.no_grad():
            probs = F.softmax(model(tensor), dim=1).squeeze().cpu().numpy()

    if thresholds is not None:
        adjusted = probs * np.array(thresholds)
        pred_idx = int(np.argmax(adjusted))
    else:
        pred_idx = int(np.argmax(probs))

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(image)
    axes[0].axis('off')
    axes[0].set_title(f'Prediction: {CLASS_NAMES[pred_idx]}\nConfidence: {probs[pred_idx]:.1%}', fontsize=12)

    colors = ['#378ADD', '#EF9F27', '#1D9E75']
    bars   = axes[1].barh(CLASS_NAMES, probs * 100, color=colors)
    axes[1].set_xlim(0, 100)
    axes[1].set_xlabel('Confidence (%)')
    axes[1].set_title('Class probabilities')
    axes[1].grid(True, axis='x', alpha=0.3)
    for bar, val in zip(bars, probs):
        axes[1].text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                     f'{val:.1%}', va='center', fontsize=10)

    plt.tight_layout()
    plt.show()
    return CLASS_NAMES[pred_idx], {CLASS_NAMES[i]: round(float(probs[i]), 4) for i in range(3)}


sample_path = test_data.iloc[0]['image_path']
true_label  = CLASS_NAMES[test_data.iloc[0]['label']]
print(f'True label: {true_label}')
pred, probs = predict_single(sample_path, use_tta=True, thresholds=best_thresholds)
print(f'Predicted : {pred}')
print(f'Probs     : {probs}')


## 12 — Batch test on your own images

In [ ]:
YOUR_TEST_FOLDER = 'my_test_images'

if os.path.isdir(YOUR_TEST_FOLDER):
    img_files = glob.glob(os.path.join(YOUR_TEST_FOLDER, '*'))
    print(f'Found {len(img_files)} images in {YOUR_TEST_FOLDER}')
    results = []
    for path in img_files:
        try:
            pred, probs = predict_single(path, use_tta=True, thresholds=best_thresholds)
            results.append({'file': os.path.basename(path), 'prediction': pred, **probs})
        except Exception as e:
            print(f'  Skipped {path}: {e}')
    df_results = pd.DataFrame(results)
    print(df_results)
    df_results.to_csv('predictions.csv', index=False)
    print('Saved to predictions.csv')
else:
    print(f'Folder "{YOUR_TEST_FOLDER}" not found.')
    print('Create it and drop your own burn images inside, then re-run this cell.')
